# 05 - Merge Landsat and Sentinel index tables

Combines the Landsat and Sentinel tables from notebook 03 (or 04) into one table per site.
Where both sensors have an observation on the same day, the Landsat value is kept; otherwise
whichever sensor is available is used.

In [ ]:
from pathlib import Path

import pandas as pd

In [ ]:
SITE = "ASP"                  # "ASP" or "BAU"
SOURCE = "fixed_footprint"    # "fixed_footprint" (notebook 03) or "ffp" (notebook 04)

VI_DIR = Path("../data/processed/vi_tables") / SOURCE
fp_landsat = VI_DIR / f"Landsat_{SITE}.csv"
fp_sentinel = VI_DIR / f"Sentinel_{SITE}.csv"
fp_out = VI_DIR / f"merged_{SITE}.csv"

In [ ]:
def parse_date(series):
    # handles 'YYYY-MM-DD' and the older 'LYYYY-MM-DD' / 'SYYYY-MM-DD' style
    core = series.astype(str).str.strip().str.extract(r"^[LS]?(\d{4}-\d{2}-\d{2})$", expand=False)
    return pd.to_datetime(core, format="%Y-%m-%d", errors="coerce")


def load(path):
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    df = df.drop(columns=[c for c in df.columns if c.startswith("Unnamed")])
    df["Date"] = parse_date(df["Date"])
    df = df.dropna(subset=["Date"])
    # more than one scene on the same day (tile overlap) -> average them
    return df.groupby("Date", as_index=False).mean(numeric_only=True)


landsat = load(fp_landsat)
sentinel = load(fp_sentinel)
vi_cols = sorted(set(landsat.columns).union(sentinel.columns) - {"Date"})

merged = (
    pd.merge(landsat, sentinel, on="Date", how="outer", suffixes=("_landsat", "_sentinel"))
    .sort_values("Date", ignore_index=True)
)

for col in vi_cols:
    l_col = f"{col}_landsat" if f"{col}_landsat" in merged.columns else (col if col in landsat.columns else None)
    s_col = f"{col}_sentinel" if f"{col}_sentinel" in merged.columns else (col if col in sentinel.columns else None)
    if l_col and s_col:
        merged[col] = merged[l_col].combine_first(merged[s_col])
    else:
        merged[col] = merged[l_col or s_col]

out = merged[["Date"] + vi_cols].dropna(subset=vi_cols, how="all")
out.to_csv(fp_out, index=False)
print(f"{len(out)} dates, {len(vi_cols)} index columns -> {fp_out}")